# Clustering of RNA Structures

This notebook provides a starting point for the different tasks involved in clustering RNA secondary structures.

## Tasks covered
1. **Data loading** – load RNA sequences and secondary structures
2. **Feature extraction** – convert dot-bracket notation into numerical feature vectors
3. **Pairwise distance computation** – compare structures using edit distance and base-pair distance
4. **Clustering** – apply K-means and hierarchical (agglomerative) clustering
5. **Evaluation** – measure cluster quality with silhouette score and dendrogram
6. **Visualisation** – PCA projection and heatmap of the distance matrix

## 1. Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform

%matplotlib inline
sns.set_theme(style="whitegrid")

## 2. Data loading

RNA secondary structures are commonly stored in **dot-bracket** notation alongside the nucleotide sequence.
Replace the sample data below with your own dataset (e.g. from a FASTA/Stockholm file or a database query).

In [ ]:
# Sample dataset: (name, sequence, dot-bracket structure)
records = [
    ("RNA_01", "GCGAUAGC",  "(((....)))"),
    ("RNA_02", "GCGAUAGC",  "((.....))"),
    ("RNA_03", "AUCGCUAGC", "((((...))))"),
    ("RNA_04", "GCAUCGAU",  "...((...))"),
    ("RNA_05", "AUGGCUAG",  "((.......))"),
    ("RNA_06", "CGAUGCAU",  ".((....))."),
    ("RNA_07", "GCUAGCAU",  "((((....))))"),
    ("RNA_08", "AUGCGCUA",  "........"),
    ("RNA_09", "GCAUGCAU",  "(.(...).)."),
    ("RNA_10", "CGAUCGUA",  "(((....)))"),
]

df = pd.DataFrame(records, columns=["name", "sequence", "structure"])
print(f"Loaded {len(df)} structures")
df.head()

## 3. Feature extraction

We encode each dot-bracket string as a numeric vector:
- `(` → 1
- `)` → -1
- `.` → 0

Sequences of different lengths are zero-padded to the maximum length.

In [ ]:
ENCODING = {"(": 1, ")": -1, ".": 0}

def encode_structure(struct: str, max_len: int) -> np.ndarray:
    """Encode a dot-bracket string to a zero-padded numeric array."""
    vec = [ENCODING.get(c, 0) for c in struct]
    vec += [0] * (max_len - len(vec))  # zero-pad
    return np.array(vec, dtype=float)

max_len = df["structure"].str.len().max()
X = np.vstack([encode_structure(s, max_len) for s in df["structure"]])

print(f"Feature matrix shape: {X.shape}")

## 4. Pairwise distance computation

Two common distances for RNA structures:
- **Euclidean distance** on the encoded vectors (proxy)
- **Edit distance** (string edit / Levenshtein) on the raw dot-bracket strings

In [ ]:
# --- Euclidean distance matrix ---
dist_euclidean = squareform(pdist(X, metric="euclidean"))

# --- String edit distance (Levenshtein) ---
def edit_distance(s1: str, s2: str) -> int:
    """Compute the Levenshtein edit distance between two strings."""
    m, n = len(s1), len(s2)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[j] = prev[j - 1]
            else:
                dp[j] = 1 + min(prev[j], dp[j - 1], prev[j - 1])
    return dp[n]

structs = df["structure"].tolist()
n = len(structs)
dist_edit = np.zeros((n, n))
for i in range(n):
    for j in range(i + 1, n):
        d = edit_distance(structs[i], structs[j])
        dist_edit[i, j] = d
        dist_edit[j, i] = d

print("Euclidean distance matrix:")
print(np.round(dist_euclidean, 2))

## 5. Clustering

### 5a. K-means clustering (on encoded feature vectors)

In [ ]:
N_CLUSTERS = 3  # adjust based on your data

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init="auto")
labels_kmeans = kmeans.fit_predict(X)

df["cluster_kmeans"] = labels_kmeans
print("K-means cluster assignments:")
print(df[["name", "structure", "cluster_kmeans"]].to_string(index=False))

### 5b. Hierarchical (agglomerative) clustering (on edit distance matrix)

In [ ]:
agg = AgglomerativeClustering(
    n_clusters=N_CLUSTERS,
    metric="precomputed",
    linkage="average",
)
labels_agg = agg.fit_predict(dist_edit)

df["cluster_hierarchical"] = labels_agg
print("Hierarchical cluster assignments:")
print(df[["name", "structure", "cluster_hierarchical"]].to_string(index=False))

## 6. Evaluation

### 6a. Silhouette score

In [ ]:
sil_kmeans = silhouette_score(X, labels_kmeans)
sil_agg    = silhouette_score(dist_edit, labels_agg, metric="precomputed")

print(f"Silhouette score – K-means (Euclidean features): {sil_kmeans:.3f}")
print(f"Silhouette score – Hierarchical (edit distance): {sil_agg:.3f}")

### 6b. Choosing the number of clusters – elbow method

In [ ]:
inertias = []
k_range = range(2, min(len(df), 8))
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(list(k_range), inertias, marker="o")
plt.xlabel("Number of clusters k")
plt.ylabel("Inertia")
plt.title("Elbow method – K-means")
plt.tight_layout()
plt.show()

## 7. Visualisation

### 7a. PCA projection coloured by K-means cluster

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)

plt.figure(figsize=(6, 5))
scatter = plt.scatter(
    X_2d[:, 0], X_2d[:, 1],
    c=labels_kmeans, cmap="tab10", s=80, edgecolors="k"
)
for i, name in enumerate(df["name"]):
    plt.annotate(name, (X_2d[i, 0] + 0.02, X_2d[i, 1] + 0.02), fontsize=8)
plt.colorbar(scatter, label="Cluster (K-means)")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("PCA of RNA structure features")
plt.tight_layout()
plt.show()

### 7b. Heatmap of the edit-distance matrix

In [ ]:
plt.figure(figsize=(7, 6))
sns.heatmap(
    dist_edit,
    xticklabels=df["name"],
    yticklabels=df["name"],
    annot=True, fmt=".0f",
    cmap="YlOrRd"
)
plt.title("Pairwise edit-distance matrix")
plt.tight_layout()
plt.show()

### 7c. Dendrogram (hierarchical clustering)

In [ ]:
condensed = squareform(dist_edit)
Z = linkage(condensed, method="average")

plt.figure(figsize=(8, 4))
dendrogram(Z, labels=df["name"].tolist(), leaf_rotation=45)
plt.title("Dendrogram – hierarchical clustering (edit distance)")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()

## 8. Next steps

- Replace the sample data with a real RNA structure dataset (e.g. from [RNAcentral](https://rnacentral.org) or [PDB](https://www.rcsb.org)).
- Implement structure-aware distances such as the **base-pair distance** or **tree edit distance** using tools like `RNAdistance` (ViennaRNA package).
- Explore additional clustering methods (DBSCAN, spectral clustering) suited to non-Euclidean distances.
- Validate clusters against known RNA families or functional annotations.